First experment, described in real\science\experiment_1_celebrities\exprement_description_link.txt

Requirments to run: HF_TOKEN setted

## 1. Data loading

### Load information to model

In [126]:
from pydantic import BaseModel
class RawSentenceData(BaseModel):
    sentence:str
    name_start_index:int
    name_end_index:int
    name:str
    

In [127]:
import json
import re

# Load a
with open(r"data\a_about_celebrities.json", "r", encoding="utf-8") as f:
    a_about_celebrities_dict = json.load(f)

a_about_celebrities: list[RawSentenceData] = []
for info in a_about_celebrities_dict:
    end_of_first_name_index = info["sentence"].find(" ", info["start_of_name_index"] + 1)
    end_of_last_name_index = re.search(r" |'s",info["sentence"][end_of_first_name_index+1:]).start() + end_of_first_name_index+1
    a_about_celebrities.append(
        RawSentenceData(
            sentence=info["sentence"],
            name_start_index=info["start_of_name_index"],
            name_end_index=end_of_last_name_index,
            name=info["sentence"][info["start_of_name_index"] : end_of_last_name_index],
        )
    )


# Load b
with open(r"data\b_mentions_celebrities.json", "r", encoding="utf-8") as f:
    b_mentions_celebrities_dict = json.load(f)

b_mentions_celebrities: list[RawSentenceData] = []
for info in b_mentions_celebrities_dict:
    end_of_first_name_index = info["sentence"].find(" ", info["start_of_name_index"] + 1)
    end_of_last_name_index = re.search(r" |'s",info["sentence"][end_of_first_name_index+1:]).start() + end_of_first_name_index+1
    b_mentions_celebrities.append(
        RawSentenceData(
            sentence=info["sentence"],
            name_start_index=info["start_of_name_index"],
            name_end_index=end_of_last_name_index,
            name=info["sentence"][info["start_of_name_index"] : end_of_last_name_index],
        )
    )

### Verify names indexes data

In [128]:
print("A data:")
for sentece_info in a_about_celebrities:
    print(f"Sentence: {sentece_info.sentence}| Name: {sentece_info.name}")
    
print("--------------------------------------------------------------")
print("B data")
for sentece_info in b_mentions_celebrities:
    print(f"Sentence: {sentece_info.sentence}| Name: {sentece_info.name}")

A data:
Sentence: Taylor Swift kicked off her world tour with a three-hour setlist spanning every era of her career.| Name: Taylor Swift
Sentence: Did you see that Dwayne Johnson posted a new workout video this morning?| Name: Dwayne Johnson
Sentence: In a statement released Thursday, Emma Watson announced her return to acting after a brief hiatus.| Name: Emma Watson
Sentence: Honestly, I think Ryan Reynolds is one of the funniest people in Hollywood right now.| Name: Ryan Reynolds
Sentence: Serena Williams retired from professional tennis after an illustrious career spanning over two decades.| Name: Serena Williams
Sentence: Breaking: Elon Musk's latest venture has sparked intense debate among industry analysts.| Name: Elon Musk
Sentence: My mom absolutely adores Tom Hanks — she's seen every single one of his movies.| Name: Tom Hanks
Sentence: Beyoncé Knowles surprised fans with an unannounced album drop late last night.| Name: Beyoncé Knowles
Sentence: According to sources close to t

### Load generated fake celebrities names

In [129]:
import random
with open(r"data\fake_celebrities_62_tokens.json", "r", encoding="utf-8") as f:
    alternative_names = json.load(f)
random.seed(1321)
random.shuffle(alternative_names)

### Show sums of token number are equal

#### Setup

In [130]:
from dotenv import load_dotenv
import os
load_dotenv(r"..\..\..\.env.local")
HF_TOKEN = os.getenv("HF_TOKEN")

In [131]:
MODEL = "meta-llama/Llama-3.1-8B"

In [132]:
from pathlib import Path
from api_checks.api_cache import ModelAPICache

model_api_cache = ModelAPICache(cache_path=Path(".model_api_cache"),hf_token=HF_TOKEN)
model_calcutor = model_api_cache.get_infomration_calculator(MODEL)

#### Calculate sum tokens amount

In [133]:
def calc_sum_name_tokens_amount(raw_sentences_data:list[RawSentenceData])->int:
    names_total_tokens = 0
    for sentence_info in raw_sentences_data:
        tokens_num = len(model_calcutor.calc_tokens(sentence_info.name)) -1 #Dont include BOS
        names_total_tokens += tokens_num
    return names_total_tokens
total_names_tokens_a = calc_sum_name_tokens_amount(a_about_celebrities)
total_names_tokens_b = calc_sum_name_tokens_amount(b_mentions_celebrities)

total_fake_names_tokens = 0
for name in alternative_names:
    tokens_num = len(model_calcutor.calc_tokens(name)) -1 #Dont include BOS
    total_fake_names_tokens += tokens_num
    

In [136]:
assert total_names_tokens_a == total_names_tokens_b == total_fake_names_tokens
print(f"All total tokens number in names are equal to {total_names_tokens_a}")

All total tokens number in names are equal to 62


## 2. Trace Calculations

### Replace information in sentences

In [ ]:
a_sentences = 